# Clustering integral de artistas musicales — workflow completo
**"Streaming, redes sociales y conciertos: construcción empírica del éxito de un artista musical"**  
Autora: Silvana Contreras — Mayo 2026

**Métodos:** K-Medoids (PAM)  ·  GMM (full & tied)  ·  Spectral Clustering (kNN)  ·  RF Feature Importance  
**Visualización:** UMAP 2D (no lineal)  
**Datasets:** 7 000 artistas para clustering · 4 913 con dato de shows para interpretación

## 0. Setup

In [ ]:
# import sys
# !{sys.executable} -m pip install umap-learn -q


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Si umap-learn no está instalado, descomentar:
#! pip install umap-learn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import cdist, squareform
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.cluster import SpectralClustering
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (silhouette_score,
                              davies_bouldin_score,
                              calinski_harabasz_score)
import warnings
warnings.filterwarnings('ignore')

try:
    import umap
    UMAP_OK = True
    print('umap-learn OK')
except ImportError:
    UMAP_OK = False
    print('umap-learn no disponible — descomenta !pip install umap-learn y reinicia kernel')

plt.rcParams['figure.dpi'] = 110
SEED = 42
print('Imports OK')

## 1. Carga de datos

In [4]:
df = pd.read_csv('dataset_7000_53.csv')
print(f'Shape: {df.shape}')
print(f"n_shows_24_25 — con dato: {df['n_shows_24_25'].notna().sum()}  "
      f"sin dato: {df['n_shows_24_25'].isna().sum()}")
df[['artist_name','sp_monthly_listeners','ins_followers','n_shows_24_25']].head(3)

Shape: (7000, 53)
n_shows_24_25 — con dato: 4913  sin dato: 2087


,artist_name,sp_monthly_listeners,ins_followers,n_shows_24_25
0,Bad Bunny,112734492.0,54721804.0,45.0
1,Taylor Swift,102733014.0,280531827.0,49.0
2,Bruno Mars,134166866.0,42922145.0,55.0


## 2. Selección de variables

**Criterios de exclusión**
- Nulos > 25 %: `tiktok_followers`, `twitter_followers`, `youtube_daily/monthly_video_views`
- Alta multicolinealidad (r > 0.80 en log1p): `sp_followers` (r=0.76 con sp_monthly_listeners),
  `num_sp_playlists` (r=0.80), `shazam_count` (r=0.90), `num_am_playlists` (r=0.81)

Se conserva una variable por dimensión conceptual.
Pares que quedan con r 0.70–0.78 representan dimensiones distintas y se aceptan.

In [ ]:
BASE_VARS = {
    'streaming_spotify':  'sp_monthly_listeners',
    'playlist_editorial': 'sp_editorial_playlist_total_reach',
    'playlist_amazon':    'num_az_playlists',
    'playlist_youtube':   'num_yt_editorial_playlists',
    'streaming_pandora':  'pandora_lifetime_streams',
    'streaming_deezer':   'deezer_fans',
    'social_instagram':   'ins_followers',
    'video_youtube':      'ycs_views',
    'tiktok_engagement':  'tiktok_top_video_comments',
}
BASE_COLS = list(BASE_VARS.values())

rows = []
for dim, col in BASE_VARS.items():
    null_pct = df[col].isna().mean() * 100
    corr_shows = (
        np.log1p(df[col].fillna(0))
          .corr(np.log1p(df['n_shows_24_25'].fillna(0)))
    )
    rows.append({'dimension': dim, 'variable': col,
                 'null_%': round(null_pct, 1),
                 'corr_log1p_shows': round(corr_shows, 3)})

print(pd.DataFrame(rows).to_string(index=False))

## 3. Imputación de nulos

| Estrategia | Variables | Criterio |
|---|---|---|
| **0** | playlist counts, pandora, deezer, tiktok_engagement | ausencia de presencia en plataforma |
| **mediana** | sp_monthly_listeners, ins_followers, ycs_views | dato faltante por cobertura incompleta |

In [ ]:
IMPUTE_ZERO = [
    'sp_editorial_playlist_total_reach', 'num_az_playlists',
    'num_yt_editorial_playlists', 'pandora_lifetime_streams',
    'deezer_fans', 'tiktok_top_video_comments',
]
IMPUTE_MED = ['sp_monthly_listeners', 'ins_followers', 'ycs_views']

df_imp = df.copy()
for col in IMPUTE_ZERO:
    df_imp[col] = df_imp[col].fillna(0)
for col in IMPUTE_MED:
    df_imp[col] = df_imp[col].fillna(df_imp[col].median())

remaining = df_imp[BASE_COLS].isnull().sum()
print('Nulos restantes tras imputación:')
print(remaining[remaining > 0] if remaining.any() else 'Ninguno — matriz limpia')

## 4. Feature engineering: log1p + ratio features + escalado

Se añaden **dos ratio features** que capturan la *forma del perfil* independientemente de la escala:  
- `social_sp_ratio` — balance redes sociales (Instagram) vs audiencia Spotify  
- `tiktok_sp_ratio` — engagement TikTok relativo a la audiencia Spotify  

Se descartaron `pandora_sp_ratio` (r=0.84 con `pandora_lifetime_streams`) y `youtube_sp_ratio`
(r=0.91 con `social_sp_ratio`) por multicolinealidad con variables ya presentes.  
El set final tiene 11 features; los únicos pares con r > 0.70 representan dimensiones conceptualmente distintas.

In [ ]:
# Log1p sobre variables base
df_log = df_imp[BASE_COLS].apply(np.log1p).copy()

# Sólo dos ratios no redundantes con las variables base
sp_anchor = df_log['sp_monthly_listeners'] + 1e-6

df_log['social_sp_ratio'] = df_log['ins_followers']             / sp_anchor
df_log['tiktok_sp_ratio'] = df_log['tiktok_top_video_comments'] / sp_anchor

RATIO_COLS = ['social_sp_ratio', 'tiktok_sp_ratio']
ALL_FEATS = BASE_COLS + RATIO_COLS

# Standard scaling
scaler = StandardScaler()
X_arr = scaler.fit_transform(df_log[ALL_FEATS])
X_sc  = pd.DataFrame(X_arr, columns=ALL_FEATS, index=df.index)

print(f'Feature matrix: {X_sc.shape}')
print(f'Features: {ALL_FEATS}')
X_sc.describe().round(2)

In [ ]:
# Verificar multicolinealidad en el set final
corr_mat = X_sc[ALL_FEATS].corr()
high = []
for i, c1 in enumerate(ALL_FEATS):
    for j, c2 in enumerate(ALL_FEATS):
        if i < j and abs(corr_mat.loc[c1, c2]) > 0.70:
            high.append({'var1': c1, 'var2': c2,
                         'r': round(corr_mat.loc[c1, c2], 3)})
if high:
    print('Pares con |r| > 0.70:')
    print(pd.DataFrame(high).to_string(index=False))
else:
    print('No hay pares con |r| > 0.70 en el set final')

## 5. UMAP — reducción dimensional no lineal (visualización)

UMAP preserva estructura local y global sin suponer linealidad.  
Se usa sólo para visualización; el clustering opera sobre la matriz completa de 11 features.

In [ ]:
if not UMAP_OK:
    print('Instalar umap-learn y reiniciar kernel')
else:
    reducer = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.1,
                        metric='euclidean', random_state=SEED, n_jobs=1)
    emb = reducer.fit_transform(X_sc.values)
    df_umap = pd.DataFrame(emb, columns=['UMAP1', 'UMAP2'], index=df.index)
    print(f'Embedding: {df_umap.shape}')

    has_shows = df['n_shows_24_25'].notna()
    shows_log = np.log1p(df.loc[has_shows, 'n_shows_24_25'])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sc1 = axes[0].scatter(
        df_umap.loc[has_shows, 'UMAP1'], df_umap.loc[has_shows, 'UMAP2'],
        c=shows_log, cmap='YlOrRd', s=3, alpha=0.6)
    plt.colorbar(sc1, ax=axes[0], label='log1p(shows 24-25)')
    axes[0].set_title('UMAP — intensidad live')
    axes[0].set_xlabel('UMAP1'); axes[0].set_ylabel('UMAP2')

    sp_log = np.log1p(df['sp_monthly_listeners'].fillna(0))
    sc2 = axes[1].scatter(
        df_umap['UMAP1'], df_umap['UMAP2'],
        c=sp_log, cmap='Blues', s=3, alpha=0.5)
    plt.colorbar(sc2, ax=axes[1], label='log1p(sp_monthly_listeners)')
    axes[1].set_title('UMAP — escala Spotify')
    axes[1].set_xlabel('UMAP1'); axes[1].set_ylabel('UMAP2')

    plt.tight_layout()
    plt.show()

## 6. Método 1: K-Medoids (PAM)

Más robusto que K-Means frente a outliers: usa puntos reales como centros de cluster.  
Implementación propia con distancia euclidiana y k-means++ initialization.  
La matriz de distancias (7 000 × 7 000) se precalcula una sola vez y se reutiliza para todos los valores de k.

In [ ]:
def kmedoids_fit(D, k, n_init=5, max_iter=100, random_state=42):
    # K-Medoids sobre matriz de distancias D precalculada (float32)
    rng = np.random.RandomState(random_state)
    n = D.shape[0]
    best_inertia = np.inf
    best_labels, best_medoids = None, None

    for _ in range(n_init):
        # k-means++ init
        medoids = [int(rng.randint(0, n))]
        for __ in range(k - 1):
            d_min = D[:, medoids].min(axis=1).astype(np.float64)
            probs = d_min ** 2
            probs /= probs.sum()
            medoids.append(int(rng.choice(n, p=probs)))

        for ___ in range(max_iter):
            labels = np.argmin(D[:, medoids], axis=1)
            new_medoids = []
            for c in range(k):
                idxs = np.where(labels == c)[0]
                if len(idxs) == 0:
                    taken = set(new_medoids)
                    candidates = [i for i in range(n) if i not in taken]
                    new_medoids.append(int(rng.choice(candidates)) if candidates else medoids[c])
                    continue
                sub_sum = D[np.ix_(idxs, idxs)].sum(axis=1)
                new_medoids.append(int(idxs[sub_sum.argmin()]))
            if sorted(new_medoids) == sorted(medoids):
                break
            medoids = new_medoids

        labels = np.argmin(D[:, medoids], axis=1)
        inertia = float(D[np.arange(n), [medoids[l] for l in labels]].sum())
        if inertia < best_inertia:
            best_inertia = inertia
            best_labels  = labels.copy()
            best_medoids = medoids[:]

    return best_labels, best_medoids

print('kmedoids_fit() definida')

In [ ]:
# Precomputar matriz de distancias una sola vez (float32 ~ 196 MB)
print('Calculando matriz 7000×7000 (puede tardar ~15 s)...')
D = cdist(X_sc.values, X_sc.values, metric='euclidean').astype(np.float32)
print(f'D shape: {D.shape}  |  memoria: {D.nbytes/1e6:.0f} MB')

In [ ]:
km_res = {}
for k in [2, 3, 4, 5]:
    print(f'K-Medoids k={k}...')
    labels, medoids = kmedoids_fit(D, k, n_init=5, max_iter=100)
    sil = silhouette_score(X_sc.values, labels)
    db  = davies_bouldin_score(X_sc.values, labels)
    ch  = calinski_harabasz_score(X_sc.values, labels)
    sizes = dict(zip(*np.unique(labels, return_counts=True)))
    print(f'  sil={sil:.4f}  DB={db:.4f}  CH={ch:.0f}  sizes={sizes}')
    km_res[k] = dict(labels=labels, medoids=medoids,
                     sil=sil, db=db, ch=ch, sizes=sizes)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
ks = list(km_res.keys())

axes[0].plot(ks, [km_res[k]['sil'] for k in ks], 'o-', color='steelblue')
axes[0].set_title('Silhouette (mayor = mejor)'); axes[0].set_xlabel('k')

axes[1].plot(ks, [km_res[k]['db'] for k in ks], 'o-', color='tomato')
axes[1].set_title('Davies-Bouldin (menor = mejor)'); axes[1].set_xlabel('k')

axes[2].plot(ks, [km_res[k]['ch'] for k in ks], 'o-', color='seagreen')
axes[2].set_title('Calinski-Harabasz (mayor = mejor)'); axes[2].set_xlabel('k')

plt.suptitle('K-Medoids — métricas internas por k', fontsize=12)
plt.tight_layout(); plt.show()

## 7. Método 2: Gaussian Mixture Models (GMM)

Permite clusters de forma elipsoidal y asignaciones probabilísticas.  
Se evalúan dos estructuras de covarianza (`full` y `tied`) y se selecciona k por BIC.

In [ ]:
gmm_res = {}
for cov in ['full', 'tied']:
    gmm_res[cov] = {}
    for k in [2, 3, 4, 5]:
        gm = GaussianMixture(n_components=k, covariance_type=cov,
                             n_init=5, max_iter=300, random_state=SEED)
        gm.fit(X_sc.values)
        labels = gm.predict(X_sc.values)
        bic = gm.bic(X_sc.values)
        aic = gm.aic(X_sc.values)
        sil = silhouette_score(X_sc.values, labels)
        db  = davies_bouldin_score(X_sc.values, labels)
        ch  = calinski_harabasz_score(X_sc.values, labels)
        sizes = dict(zip(*np.unique(labels, return_counts=True)))
        print(f'GMM [{cov}] k={k}: BIC={bic:.0f}  AIC={aic:.0f}  '
              f'sil={sil:.4f}  DB={db:.4f}  sizes={sizes}')
        gmm_res[cov][k] = dict(labels=labels, model=gm,
                               bic=bic, aic=aic,
                               sil=sil, db=db, ch=ch, sizes=sizes)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
col_map = {'full': 'steelblue', 'tied': 'darkorange'}
ks = [2, 3, 4, 5]

for cov in ['full', 'tied']:
    bics = [gmm_res[cov][k]['bic'] for k in ks]
    aics = [gmm_res[cov][k]['aic'] for k in ks]
    axes[0].plot(ks, bics, 'o-', color=col_map[cov], label=f'cov={cov}')
    axes[1].plot(ks, aics, 'o-', color=col_map[cov], label=f'cov={cov}')

for ax, title in zip(axes, ['BIC (menor = mejor)', 'AIC (menor = mejor)']):
    ax.set_title(title); ax.set_xlabel('k'); ax.legend()

plt.suptitle('GMM — selección de k', fontsize=12)
plt.tight_layout(); plt.show()

## 8. Método 3: RF Unsupervised + Spectral Clustering

**Parte A — RF unsupervised (Shi & Horvath 2005):**  
Se entrena un Random Forest para distinguir datos reales de datos sintéticos (columnas permutadas).  
La importancia de variables resultante revela cuáles features son más estructurantes del espacio digital.  
*Nota*: la matriz de proximidad RF degenera para n=7 000 (la mayoría de pares nunca comparte hoja),
por lo que no se usa como input de clustering jerárquico.  

**Parte B — Spectral Clustering (kNN affinity):**  
Construye un grafo de vecinos más cercanos (k=30) sobre el espacio escalado y aplica clustering espectral.  
Captura relaciones no lineales de densidad local sin supuestos de forma de cluster y sin
necesidad de precalcular una matriz densa de distancias.

In [ ]:
# === RF Unsupervised: feature importance ===
rng_rf = np.random.RandomState(SEED)
X_real = X_sc.values.astype(np.float32)
n_rf, p_rf = X_real.shape

X_synth = np.column_stack([
    rng_rf.permutation(X_real[:, j]) for j in range(p_rf)
]).astype(np.float32)

X_tr = np.vstack([X_real, X_synth])
y_tr = np.array([1]*n_rf + [0]*n_rf)

rf = RandomForestClassifier(n_estimators=100, max_depth=10,
                             min_samples_leaf=5, n_jobs=-1, random_state=SEED)
rf.fit(X_tr, y_tr)

fi = pd.Series(rf.feature_importances_, index=ALL_FEATS).sort_values(ascending=False)
print('Feature importance RF (real vs sintético):')
print(fi.round(4))

fig, ax = plt.subplots(figsize=(10, 4))
fi.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('RF Feature Importance — variables más estructurantes del perfil digital')
ax.set_ylabel('Importancia'); ax.set_xlabel('')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

In [ ]:
# === Spectral Clustering (kNN affinity, no lineal) ===
sp_res = {}
for k in [2, 3, 4, 5]:
    print(f'Spectral k={k}...')
    sc = SpectralClustering(n_clusters=k, affinity='nearest_neighbors',
                            n_neighbors=30, assign_labels='kmeans',
                            n_init=10, random_state=SEED, n_jobs=-1)
    labels_sp = sc.fit_predict(X_sc.values)
    sil = silhouette_score(X_sc.values, labels_sp)
    db  = davies_bouldin_score(X_sc.values, labels_sp)
    ch  = calinski_harabasz_score(X_sc.values, labels_sp)
    sizes = dict(zip(*np.unique(labels_sp, return_counts=True)))
    min_pct = 100 * min(sizes.values()) / len(df)
    print(f'  sil={sil:.4f}  DB={db:.4f}  CH={ch:.0f}  min%={min_pct:.1f}  sizes={sizes}')
    sp_res[k] = dict(labels=labels_sp, sil=sil, db=db, ch=ch, sizes=sizes)

## 9. Comparación de métodos — tabla de métricas

In [ ]:
rows_cmp = []
for k in [2, 3, 4, 5]:
    for method, res in [('K-Medoids', km_res[k]),
                         ('GMM-full',  gmm_res['full'][k]),
                         ('GMM-tied',  gmm_res['tied'][k]),
                         ('Spectral',  sp_res[k])]:
        min_size_pct = 100 * min(res['sizes'].values()) / len(df)
        rows_cmp.append({
            'method': method, 'k': k,
            'silhouette':       round(res['sil'], 4),
            'davies_bouldin':   round(res['db'],  4),
            'calinski_harabasz':round(res['ch'],  1),
            'min_cluster_%':    round(min_size_pct, 1),
        })

cmp_df = pd.DataFrame(rows_cmp)
# Usable: sil > 0.15 AND min cluster >= 4% (280+ artistas)
cmp_df['usable'] = (cmp_df['silhouette'] > 0.15) & (cmp_df['min_cluster_%'] >= 4.0)
print(cmp_df.sort_values(['method','k']).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = {'K-Medoids':'steelblue','GMM-full':'darkorange',
          'GMM-tied':'gold','Spectral':'forestgreen'}
ks = [2, 3, 4, 5]

for ax, metric, title in zip(
        axes,
        ['silhouette','davies_bouldin','calinski_harabasz'],
        ['Silhouette (↑)','Davies-Bouldin (↓)','Calinski-Harabasz (↑)']):
    for method, grp in cmp_df.groupby('method'):
        ax.plot(grp['k'], grp[metric], 'o-',
                color=colors[method], label=method, linewidth=1.5, markersize=6)
    ax.set_title(title); ax.set_xlabel('k'); ax.legend(fontsize=8)

plt.suptitle('Comparación de métodos — métricas internas', fontsize=12)
plt.tight_layout(); plt.show()

## 10. Selección del mejor modelo y perfiles de clusters

Criterio: mayor silhouette entre soluciones con cluster mínimo ≥ 5 % del dataset.  
Se puede sobreescribir manualmente cambiando `OVERRIDE_METHOD` / `OVERRIDE_K`.

In [ ]:
# Auto-selección: mejor silhouette entre soluciones 'usable'
OVERRIDE_METHOD = None   # ej. 'K-Medoids' para forzar método
OVERRIDE_K      = None   # ej. 3 para forzar k

usable = cmp_df[cmp_df['usable']]
if len(usable) == 0:
    print('Ninguna solución supera los umbrales — se toma la de mayor silhouette total')
    usable = cmp_df

if OVERRIDE_METHOD and OVERRIDE_K:
    best_row = cmp_df[(cmp_df['method']==OVERRIDE_METHOD) & (cmp_df['k']==OVERRIDE_K)].iloc[0]
else:
    best_row = usable.sort_values('silhouette', ascending=False).iloc[0]

BM = best_row['method']
BK = int(best_row['k'])
print(f'Mejor modelo: {BM}  k={BK}')
print(f'  Silhouette={best_row["silhouette"]}  DB={best_row["davies_bouldin"]}  '
      f'CH={best_row["calinski_harabasz"]}  min_cluster%={best_row["min_cluster_%"]}')

# Recuperar etiquetas
if BM == 'K-Medoids':
    best_labels = km_res[BK]['labels']
elif BM == 'GMM-full':
    best_labels = gmm_res['full'][BK]['labels']
elif BM == 'GMM-tied':
    best_labels = gmm_res['tied'][BK]['labels']
else:
    best_labels = sp_res[BK]['labels']

df_work      = df.copy()
df_work['cluster'] = best_labels
df_log['cluster']  = best_labels

print('\nTamaño de clusters:')
print(df_work['cluster'].value_counts().sort_index())

### 10a. Perfil digital de clusters — heatmap

In [ ]:
PROFILE_FEATS = BASE_COLS + RATIO_COLS

profile = df_log.groupby('cluster')[PROFILE_FEATS].median()

# Z-score entre clusters por feature (quién está por encima/debajo de la media)
profile_z = profile.sub(profile.mean()).div(profile.std() + 1e-9)

fig, ax = plt.subplots(figsize=(16, max(3, BK * 1.4)))
sns.heatmap(profile_z, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.4, ax=ax)
ax.set_title(f'{BM}  k={BK} — perfil digital (z-score de mediana log1p)')
ax.set_xlabel('Feature'); ax.set_ylabel('Cluster')
plt.tight_layout(); plt.show()

print('\nMedianas log1p por cluster:')
print(profile.round(2))

In [ ]:
# Box-plots de variables clave por cluster
KEY_FEATS = ['sp_monthly_listeners', 'pandora_lifetime_streams',
             'ins_followers', 'ycs_views',
             'social_sp_ratio', 'tiktok_sp_ratio']

fig, axes = plt.subplots(2, 3, figsize=(14, 7))

for ax, feat in zip(axes.flat, KEY_FEATS):
    data = [df_log.loc[df_log['cluster']==c, feat].dropna().values
            for c in sorted(df_log['cluster'].unique())]
    ax.boxplot(data, labels=[f'C{c}' for c in sorted(df_log['cluster'].unique())],
               showfliers=False)
    ax.set_title(feat, fontsize=9)
    ax.set_xlabel('Cluster')

plt.suptitle(f'Distribución de variables clave por cluster ({BM} k={BK})', fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# UMAP coloreado por clusters (sólo si UMAP corrió en sección 5)
if UMAP_OK and 'df_umap' in globals():
    palette_c = plt.cm.Set1.colors
    fig, ax = plt.subplots(figsize=(8, 6))
    for c in sorted(np.unique(best_labels)):
        mask = best_labels == c
        ax.scatter(df_umap.loc[mask, 'UMAP1'], df_umap.loc[mask, 'UMAP2'],
                   s=4, alpha=0.5,
                   color=palette_c[c % len(palette_c)], label=f'Cluster {c}')
    ax.set_title(f'UMAP coloreado por cluster — {BM} k={BK}')
    ax.set_xlabel('UMAP1'); ax.set_ylabel('UMAP2')
    ax.legend(markerscale=3)
    plt.tight_layout(); plt.show()
else:
    print('UMAP no disponible o no ejecutado — instalar umap-learn y correr sección 5 primero')

## 11. Cruce con metadata

In [ ]:
for var in ['genre_short', 'country_short', 'pronoun_short',
            'major_record_label', 'band']:
    ct = (pd.crosstab(df_work['cluster'], df_work[var],
                      normalize='index') * 100).round(1)
    print(f'\n--- {var} (% dentro del cluster) ---')
    print(ct.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4 + BK * 0.4))

genre_ct = (pd.crosstab(df_work['cluster'], df_work['genre_short'],
                        normalize='index') * 100)
genre_ct.plot(kind='bar', stacked=True, ax=axes[0],
              colormap='tab20', legend=True)
axes[0].set_title('Composición por género')
axes[0].set_xlabel('Cluster'); axes[0].set_ylabel('%')
axes[0].legend(loc='upper right', fontsize=7, ncol=2)
axes[0].tick_params(axis='x', rotation=0)

region_ct = (pd.crosstab(df_work['cluster'], df_work['country_short'],
                         normalize='index') * 100)
region_ct.plot(kind='bar', stacked=True, ax=axes[1],
               colormap='Set2', legend=True)
axes[1].set_title('Composición por región')
axes[1].set_xlabel('Cluster'); axes[1].set_ylabel('%')
axes[1].legend(loc='upper right', fontsize=8)
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout(); plt.show()

## 12. Cruce con actividad en vivo (subset 4 913)

El cluster fue asignado sobre los 7 000 artistas.  
Para la interpretación live se restringe al subconjunto con `n_shows_24_25` disponible.

In [ ]:
df_live = df_work[df_work['n_shows_24_25'].notna()].copy()
print(f'Subset con dato live: {len(df_live)} artistas')

stats_rows = []
for c in sorted(df_live['cluster'].unique()):
    grp = df_live.loc[df_live['cluster'] == c, 'n_shows_24_25']
    stats_rows.append({
        'cluster': c,
        'n': len(grp),
        'pct_zero': round(100*(grp==0).mean(), 1),
        'median':   grp.median(),
        'p25':      grp.quantile(0.25),
        'p75':      grp.quantile(0.75),
        'p90':      grp.quantile(0.90),
        'max':      grp.max()
    })

print(pd.DataFrame(stats_rows).to_string(index=False))

In [ ]:
# Boxplot shows por cluster
cluster_ids = sorted(df_live['cluster'].unique())
data_plot   = [df_live.loc[df_live['cluster']==c, 'n_shows_24_25'].values
               for c in cluster_ids]
labels_plot = [f'C{c}' for c in cluster_ids]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].boxplot(data_plot, labels=labels_plot, showfliers=False)
axes[0].set_title('n_shows_24_25 por cluster (sin outliers)')
axes[0].set_xlabel('Cluster'); axes[0].set_ylabel('Shows')

axes[1].boxplot([np.log1p(d) for d in data_plot],
                labels=labels_plot, showfliers=False)
axes[1].set_title('log1p(n_shows_24_25) por cluster')
axes[1].set_xlabel('Cluster'); axes[1].set_ylabel('log1p(shows)')

plt.suptitle(f'Actividad en vivo por cluster — {BM} k={BK}', fontsize=11)
plt.tight_layout(); plt.show()

---
## 13. Análisis detallado por cluster: K-Medoids k=3 y Spectral k=3

Se profundiza en las dos soluciones seleccionadas.  
Para cada cluster: perfil digital completo · actividad en vivo · composición de género, región, formato y sello.

In [ ]:
# === Reproducir ambos modelos ===
print("K-Medoids k=3...")
km3_labels, _ = kmedoids_fit(D, 3, n_init=5, max_iter=100)
print(f"  sizes: {dict(zip(*np.unique(km3_labels, return_counts=True)))}")

print("Spectral k=3...")
sp3_labels = SpectralClustering(
    n_clusters=3, affinity='nearest_neighbors', n_neighbors=30,
    assign_labels='kmeans', n_init=10, random_state=SEED, n_jobs=-1
).fit_predict(X_sc.values)
print(f"  sizes: {dict(zip(*np.unique(sp3_labels, return_counts=True)))}")

# Etiquetas canonicas (reordenadas por mediana de shows para interpretación consistente)
df_full = df_imp.copy()
df_full['km3_raw'] = km3_labels
df_full['sp3_raw'] = sp3_labels
df_full['n_shows_24_25'] = df['n_shows_24_25'].values

for raw_col, final_col in [('km3_raw','km3'), ('sp3_raw','sp3')]:
    med_shows = (df_full[df_full['n_shows_24_25'].notna()]
                 .groupby(raw_col)['n_shows_24_25'].median()
                 .sort_values(ascending=False))
    remap = {old: new for new, old in enumerate(med_shows.index)}
    df_full[final_col] = df_full[raw_col].map(remap)

# Adjuntar metadatos
for col in ['genre_short','country_short','pronoun_short','major_record_label','band']:
    df_full[col] = df[col].values

km3 = df_full['km3'].values
sp3 = df_full['sp3'].values
print("\nKM3 clusters (ordenados por shows desc):", dict(zip(*np.unique(km3, return_counts=True))))
print("SP3 clusters (ordenados por shows desc):", dict(zip(*np.unique(sp3, return_counts=True))))

In [ ]:
# === Tabla resumen completa por cluster y modelo ===
df_log2 = df_imp[BASE_COLS].apply(np.log1p).copy()
sp_a = df_log2['sp_monthly_listeners'] + 1e-6
df_log2['social_sp_ratio'] = df_log2['ins_followers'] / sp_a
df_log2['tiktok_sp_ratio'] = df_log2['tiktok_top_video_comments'] / sp_a

n_total = len(df)
rows_sum = []
for model_name, labels in [('K-Medoids k=3', km3), ('Spectral k=3', sp3)]:
    for c in sorted(np.unique(labels)):
        mask      = labels == c
        g         = df_full[mask]
        g_log     = df_log2[mask]
        g_live    = g[g['n_shows_24_25'].notna()]
        shows     = g_live['n_shows_24_25']
        rows_sum.append({
            'modelo':      model_name,
            'cluster':     f'C{c}',
            'n':           mask.sum(),
            'pct_total':   round(100*mask.sum()/n_total, 1),
            'major_%':     round(100*g['major_record_label'].mean(), 1),
            'band_%':      round(100*g['band'].mean(), 1),
            'n_live':      len(g_live),
            'pct_zero_shows': round(100*(shows==0).mean(), 1),
            'med_shows':   shows.median(),
            'p75_shows':   shows.quantile(.75),
            'p90_shows':   shows.quantile(.90),
            'sp_monthly_med':  round(g_log['sp_monthly_listeners'].median(), 2),
            'pandora_med':     round(g_log['pandora_lifetime_streams'].median(), 2),
            'ins_med':         round(g_log['ins_followers'].median(), 2),
            'ycs_med':         round(g_log['ycs_views'].median(), 2),
            'social_ratio_med':round(g_log['social_sp_ratio'].median(), 2),
        })

summary_df = pd.DataFrame(rows_sum)
print(summary_df.to_string(index=False))

In [ ]:
# === Heatmaps z-score lado a lado ===
PLOT_FEATS = BASE_COLS + ['social_sp_ratio']   # 10 features legibles
FEAT_LABELS = ['sp_monthly', 'sp_editorial', 'num_az_playlist',
               'num_yt_editorial', 'pandora', 'deezer',
               'instagram', 'yt_channel', 'tiktok_eng', 'social_ratio']

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

for ax, (model_name, labels) in zip(axes, [('K-Medoids k=3', km3), ('Spectral k=3', sp3)]):
    prof = {}
    for c in sorted(np.unique(labels)):
        prof[f'C{c}'] = df_log2[labels == c][PLOT_FEATS].median().values
    prof_df = pd.DataFrame(prof, index=FEAT_LABELS).T
    prof_z  = prof_df.sub(prof_df.mean()).div(prof_df.std() + 1e-9)
    sns.heatmap(prof_z, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, linewidths=0.4, ax=ax, cbar=True)
    ax.set_title(f'{model_name} — z-score mediana log1p', fontsize=11)
    ax.set_xlabel('Feature'); ax.set_ylabel('Cluster')
    ax.tick_params(axis='x', rotation=40)

plt.tight_layout()
plt.show()

In [ ]:
# === Radar / spider charts — perfil de cada cluster ===
RADAR_FEATS  = ['sp_monthly', 'sp_editorial', 'num_az_playlist',
                'num_yt_editorial', 'pandora', 'deezer',
                'instagram', 'yt_channel', 'tiktok_eng', 'social_ratio']
N_FEAT = len(RADAR_FEATS)
angles = np.linspace(0, 2*np.pi, N_FEAT, endpoint=False).tolist()
angles += angles[:1]

COLORS = ['steelblue', 'darkorange', 'forestgreen']

fig, axes = plt.subplots(1, 2, figsize=(14, 6),
                         subplot_kw=dict(polar=True))

for ax, (model_name, labels) in zip(axes, [('K-Medoids k=3', km3), ('Spectral k=3', sp3)]):
    # Compute median profile per cluster (log1p, then min-max normalize across clusters)
    prof = {}
    for c in sorted(np.unique(labels)):
        prof[c] = df_log2[labels == c][PLOT_FEATS].median().values
    prof_arr = np.array([prof[c] for c in sorted(prof)])
    # Min-max normalization per feature across clusters
    mn, mx = prof_arr.min(axis=0), prof_arr.max(axis=0)
    prof_norm = (prof_arr - mn) / (mx - mn + 1e-9)

    for i, c in enumerate(sorted(np.unique(labels))):
        vals = prof_norm[i].tolist() + [prof_norm[i][0]]
        ax.plot(angles, vals, 'o-', color=COLORS[i], linewidth=2, label=f'C{c}')
        ax.fill(angles, vals, alpha=0.08, color=COLORS[i])

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(RADAR_FEATS, size=8)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75])
    ax.set_yticklabels(['0.25','0.5','0.75'], size=7)
    ax.set_title(model_name, pad=18, fontsize=11)
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=9)

plt.suptitle('Perfil digital por cluster — normalización min-max por feature', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# === Actividad en vivo: violins lado a lado ===
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (model_name, labels, col_name) in zip(
        axes,
        [('K-Medoids k=3', km3, 'km3'), ('Spectral k=3', sp3, 'sp3')]):

    df_full['_labels'] = labels
    live_data = df_full[df_full['n_shows_24_25'].notna()].copy()
    live_data['log_shows'] = np.log1p(live_data['n_shows_24_25'])
    cluster_ids = sorted(live_data['_labels'].unique())

    parts = ax.violinplot(
        [live_data.loc[live_data['_labels']==c, 'log_shows'].values for c in cluster_ids],
        positions=cluster_ids, showmedians=True, showextrema=False)

    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(COLORS[i % 3])
        pc.set_alpha(0.6)
    parts['cmedians'].set_color('black')
    parts['cmedians'].set_linewidth(2)

    # Overlay median text
    for c in cluster_ids:
        med = live_data.loc[live_data['_labels']==c, 'n_shows_24_25'].median()
        ax.text(c, live_data.loc[live_data['_labels']==c,'log_shows'].max() + 0.15,
                f'med={med:.0f}', ha='center', fontsize=9, color=COLORS[c % 3])

    ax.set_xticks(cluster_ids)
    ax.set_xticklabels([f'C{c}' for c in cluster_ids])
    ax.set_xlabel('Cluster'); ax.set_ylabel('log1p(n_shows_24_25)')
    ax.set_title(f'{model_name} — distribución shows live')

plt.suptitle('Actividad en vivo por cluster (artistas con dato)', fontsize=12)
plt.tight_layout(); plt.show()

# Tabla complementaria: % sin shows y % con >50 shows
for model_name, labels in [('K-Medoids k=3', km3), ('Spectral k=3', sp3)]:
    df_full['_labels'] = labels
    live_data = df_full[df_full['n_shows_24_25'].notna()]
    print(f"\n{model_name}:")
    for c in sorted(live_data['_labels'].unique()):
        g = live_data.loc[live_data['_labels']==c,'n_shows_24_25']
        print(f"  C{c} n={len(g):4d}  zero={100*(g==0).mean():4.1f}%  "
              f">10shows={100*(g>10).mean():4.1f}%  >50shows={100*(g>50).mean():4.1f}%  "
              f"med={g.median():4.0f}  p90={g.quantile(.9):4.0f}")

In [ ]:
# === Composición por género y región — ambos modelos ===
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for row_idx, (model_name, labels) in enumerate([('K-Medoids k=3', km3), ('Spectral k=3', sp3)]):
    df_full['_labels'] = labels

    # Género
    g_ct = (pd.crosstab(df_full['_labels'], df_full['genre_short'], normalize='index') * 100)
    g_ct.index = [f'C{i}' for i in g_ct.index]
    g_ct.plot(kind='bar', stacked=True, ax=axes[row_idx][0],
              colormap='tab20', legend=(row_idx==0))
    axes[row_idx][0].set_title(f'{model_name} — Género')
    axes[row_idx][0].set_xlabel('Cluster'); axes[row_idx][0].set_ylabel('%')
    axes[row_idx][0].tick_params(axis='x', rotation=0)
    if row_idx == 0:
        axes[row_idx][0].legend(loc='upper right', fontsize=7, ncol=2)

    # Región
    r_ct = (pd.crosstab(df_full['_labels'], df_full['country_short'], normalize='index') * 100)
    r_ct.index = [f'C{i}' for i in r_ct.index]
    r_ct.plot(kind='bar', stacked=True, ax=axes[row_idx][1],
              colormap='Set2', legend=(row_idx==0))
    axes[row_idx][1].set_title(f'{model_name} — Región')
    axes[row_idx][1].set_xlabel('Cluster'); axes[row_idx][1].set_ylabel('%')
    axes[row_idx][1].tick_params(axis='x', rotation=0)
    if row_idx == 0:
        axes[row_idx][1].legend(loc='upper right', fontsize=8)

plt.suptitle('Composición de género y región por cluster', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# === Major label, banda/solista y pronombres — ambos modelos ===
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

bar_w  = 0.35
x      = np.arange(3)   # 3 clusters

for ax, var, title, ylab in zip(
        axes,
        ['major_record_label', 'band', 'pronoun_short'],
        ['Major Record Label (%)', 'Formato: Banda (%)', 'Pronombres (% he/him)'],
        ['% artistas', '% artistas', '% artistas']):

    all_vals = []
    for offset, (model_name, labels, color) in enumerate(
            [('K-Medoids k=3', km3, 'steelblue'), ('Spectral k=3', sp3, 'darkorange')]):
        df_full['_labels'] = labels
        vals = []
        for c in sorted(np.unique(labels)):
            g = df_full[df_full['_labels'] == c]
            if var == 'pronoun_short':
                vals.append(100 * (g[var] == 'he/him').mean())
            else:
                vals.append(100 * g[var].mean())
        ax.bar(x + offset*bar_w, vals, width=bar_w, label=model_name,
               color=color, alpha=0.85)
        all_vals.extend(vals)

    ax.set_xticks(x + bar_w/2)
    ax.set_xticklabels(['C0','C1','C2'])
    ax.set_title(title); ax.set_ylabel(ylab)
    ax.legend(fontsize=8)
    ax.set_ylim(0, max(all_vals) * 1.15 + 2)

plt.suptitle('Metadata estructural por cluster', fontsize=12)
plt.tight_layout(); plt.show()

# Tabla de pronombres completa
print("\nDistribución de pronombres por cluster (%):")
for model_name, labels in [('K-Medoids k=3', km3), ('Spectral k=3', sp3)]:
    df_full['_labels'] = labels
    ct = (pd.crosstab(df_full['_labels'], df_full['pronoun_short'], normalize='index') * 100).round(1)
    ct.index = [f'C{i}' for i in ct.index]
    print(f"\n{model_name}:")
    print(ct.to_string())

---
## 14. Análisis escrito por cluster

### K-Medoids k=3

---

**Cluster 0 — Artistas de alta actividad consolidada (n=1 880, 27 %)**

Es el grupo más compacto y homogéneo. Concentra los artistas con los valores más altos en todas las dimensiones del ecosistema digital: la mediana de oyentes mensuales en Spotify supera los **8,9 millones** (log1p=16.0), el alcance editorial llega a **88 millones** (log1p=17.1) y las reproducciones históricas en Pandora son de **500 millones** (log1p=19.9). Esta presencia cruzada en streaming, playlists curatoriales y redes indica una integración profunda con el ecosistema musical digital anglosajón. El 43 % pertenece a un major label —el porcentaje más alto del dataset—, y la región dominante es Norteamérica (52 %). La actividad en vivo es la más alta: mediana de **24 shows** en 2024-2025, el 88 % de los artistas con dato registró al menos un show, y el p90 alcanza 78 conciertos. Pop, Hip-hop & trap y Latin & urban son los géneros líderes.

*Interpretación*: artistas con infraestructura industrial completa. El streaming masivo y la presencia en playlists editoriales coexisten con una alta actividad live, lo que refuerza la hipótesis de complementariedad entre las dimensiones digitales y presenciales.

---

**Cluster 1 — Artistas mid-tier globalmente diversificados (n=4 419, 63 %)**

Es el cluster mayoritario y actúa como el "gran centro" del ecosistema digital. Los valores en todas las métricas son intermedios: 2,8 millones de oyentes mensuales (log1p=14.8), alcance editorial de 5,3 millones (log1p=15.5) y Pandora en 10,4 millones (log1p=16.1). La ratio social/streaming (0.87) es la más baja, indicando que la audiencia de Spotify es relativamente grande respecto a la presencia social. El 22 % está en un major, y la distribución geográfica es la más diversificada: Norteamérica (36 %), LATAM (18 %), Europa sin GB (17 %). La actividad live es moderada: mediana de **9 shows**, con un 80 % de artistas que registra al menos un show. Los géneros Electronic & dance y Pop tienen peso relevante.

*Interpretación*: artistas establecidos en múltiples plataformas pero sin el nivel de consolidación industrial del C0. Representan el grueso de la industria global, con presencia digital transversal y actividad live sostenida pero no intensa.

---

**Cluster 2 — Artistas social-video nativos, bajo anclaje en plataformas de streaming (n=701, 10 %)**

Es el cluster más singular. Tiene los valores más bajos en playlists (num_az=0, num_yt_editorial=0) y en Pandora (log1p=5.1), pero los más altos en **ratio social/streaming** (1.16 vs 0.87-0.91 de los otros clusters) y en **YouTube channel** (log1p=19.7). Esto indica artistas cuya audiencia está más anclada en Instagram y YouTube que en el ecosistema de playlists y streaming. El 7 % pertenece a major labels y el 19 % son bandas —ambos los mínimos del dataset—. La región Asia & Oceanía representa el 19 %, el porcentaje más alto de los tres clusters. La actividad live es muy baja: mediana de **2 shows**, 33 % sin ningún show registrado. El p90 llega solo a 18 conciertos. Géneros Electronic & dance y K-Pop tienen sobrerrepresentación.

*Interpretación*: artistas que construyeron audiencia a través de redes sociales y video (probablemente TikTok e Instagram) sin haber alcanzado integración en el ecosistema de playlists editoriales y streaming multiplataforma. Pueden representar la "generación TikTok" o artistas de nicho regional con audiencias digitales sin conversión al mercado de conciertos.

---

### Spectral k=3

---

**Cluster 0 — Élite global integrada (n=880, 12.6 %)**

Versión más concentrada y extrema del C0 de K-Medoids. Valores máximos en todas las dimensiones: 16 millones de oyentes mensuales (log1p=16.6), alcance editorial de 47 millones (log1p=17.7), Pandora en 970 millones (log1p=20.7), Instagram con 4.9 millones de seguidores (log1p=15.3). El 51 % pertenece a major labels —el valor más alto de todo el análisis—. Norteamérica representa el 56 %. La actividad live es la más intensa del dataset: mediana de **29 shows**, solo el 10.6 % sin shows, y el p90 alcanza 83 conciertos. Pop, Hip-hop & trap y Latin & urban encabezan la distribución de géneros.

*Interpretación*: los artistas más consolidados del ecosistema global. Su perfil sugiere que la máxima integración digital —playlists, redes, streaming, video— coexiste con la máxima actividad en vivo. Este grupo materializa con mayor claridad la hipótesis de ecosistema multiplataforma.

---

**Cluster 1 — Mid-tier global masivo (n=5 820, 83 %)**

Agrupa a la gran mayoría del dataset. Perfil digital moderado en todas las dimensiones, con valores levemente por debajo del C0 pero sin un rasgo distintivo extremo. La diversidad geográfica y de género es máxima: Norteamérica (37 %), LATAM (17 %), Europa (16 %); Pop (20 %), Electronic (14 %), Hip-hop (13 %). El 24 % está en un major. Mediana de **10 shows** y 19 % sin shows. Funcionalmente, este cluster captura la estructura de la industria musical en su "nivel estándar": artistas con presencia digital establecida pero sin el nivel de integración institucional del C0.

*Interpretación*: el cluster refleja la distribución esperada en una industria concentrada: la mayoría de artistas exitosos comparte un espacio común de visibilidad moderada, sin diferencias de perfil suficientemente marcadas para formar grupos más específicos bajo este modelo.

---

**Cluster 2 — Artistas con presencia social-video sin infraestructura de streaming (n=300, 4.3 %)**

El hallazgo más específico del análisis. Tiene los valores más bajos del dataset en prácticamente todas las métricas de streaming y playlists: **cero** en editorial Spotify, cero en Amazon playlists, cero en YouTube editorial y **cero en Pandora**. Sin embargo, mantiene presencia digital en Instagram (log1p=13.7) y YouTube Channel (log1p=19.7). La **ratio social/streaming es la más alta de todos los clusters** (1.46), lo que indica que estos artistas tienen relativamente muchos seguidores sociales por cada oyente de Spotify. Solo el 4.5 % tiene major label. La actividad live es mínima: mediana de **2 shows**, 35 % sin ningún concierto. El género Latin & urban tiene mayor peso que en los otros clusters (15 %).

*Interpretación*: artistas con audiencia construida en plataformas de video y redes sociales —posiblemente TikTok, YouTube o Instagram— que no han logrado (o no han buscado) integración en el ecosistema de playlists y streaming multiplataforma. La ausencia total de presencia en Pandora y playlists editoriales sugiere o bien artistas no anglófonos que no circulan por ese sistema, o bien una generación de creadores cuyo modelo de negocio no depende del streaming tradicional. La baja actividad live podría indicar que sin el anclaje en el ecosistema de streaming y playlists, la conversión a shows presenciales es limitada.

---

### Convergencia entre modelos

Ambos modelos convergen en tres hallazgos fundamentales:

1. **La consolidación digital multiplataforma se asocia positivamente con la actividad en vivo.** El cluster de mayor integración digital (C0 en ambos modelos) siempre presenta la mediana de shows más alta. La brecha es consistente: 24-29 shows vs 9-10 vs 2.

2. **Existe un segmento de artistas con audiencia en redes sociales y video pero sin integración en el ecosistema de streaming y playlists.** Este grupo (C2 en K-Medoids, C2 en Spectral) concentra bajo nivel live, baja presencia en major labels y alta ratio social/streaming.

3. **El grueso de la industria (60-83 %) ocupa un espacio intermedio** con presencia moderada en todas las plataformas, diversidad geográfica y de género, y actividad live sostenida pero no intensa.

Estos hallazgos son consistentes con la hipótesis de **ecosistema multiplataforma**: la integración en múltiples canales digitales —streaming, playlists curatoriales, redes sociales— aparece asociada a mayor tracción en el mercado de conciertos, mientras que la presencia exclusivamente social/video sin anclaje en streaming no se traduce en actividad live.